<a href="https://colab.research.google.com/github/AditiKrr/DRN_DSA-FDNet-timeseries-forecasting/blob/drnfd/hydprecip_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **DRN DSA**

DRN → learns useful patterns/features through residual neural-network layers
DSA → learns a compressed representation of the input features
Fusion → combines the information from both → predicts precipitation


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


def load_data(path="Hyderabad_Consolidated_RA.xlsx"):
    df = pd.read_excel(path, sheet_name="Sheet2")
    df = df.sort_values(["Years", "Months"]).reset_index(drop=True)

    month_map = {m: i+1 for i, m in enumerate(
        ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])}
    df["MonthNum"] = df["Months"].map(month_map)
    # cyclical encoding of month captures monsoon seasonality
    df["Month_sin"] = np.sin(2*np.pi*df["MonthNum"]/12)
    df["Month_cos"] = np.cos(2*np.pi*df["MonthNum"]/12)

    feature_cols = ["Cloud Cover", "Diurnal Temp. Range", "Vapour Pressure", "Wet day Freq."]
    target_col = "Precipitation"

    df = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)
    return df, feature_cols, target_col

if __name__ == "__main__":
    df, feats, target = load_data()
    print(df.shape)
    print(df[feats + [target]].describe())


(1224, 16)
       Cloud Cover  Diurnal Temp. Range  Vapour Pressure  Wet day Freq.  \
count  1224.000000          1224.000000      1224.000000    1224.000000   
mean     36.733222            11.681568        19.342386       4.067360   
std      23.370382             1.969728         4.486435       3.965015   
min       0.077000             8.508000        12.415000       0.000000   
25%      17.169000             9.709750        14.484000       1.000000   
50%      29.094500            11.873000        19.210000       2.357800   
75%      57.700000            13.602250        24.423000       7.223450   
max      85.769000            14.623000        27.968000      15.450500   

       Precipitation  
count     1224.00000  
mean        70.06280  
std         89.44805  
min          0.00000  
25%          2.15125  
50%         25.01950  
75%        117.80375  
max        544.26100  


In [ ]:
%%writefile /content/data_prep.py

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def load_data(path="/content/Hyderabad_Consolidated_RA.xlsx"):
    df = pd.read_excel(path, sheet_name="Sheet2")
    df = df.sort_values(["Years", "Months"]).reset_index(drop=True)

    month_map = {
        m: i + 1 for i, m in enumerate(
            ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
             "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
        )
    }

    df["MonthNum"] = df["Months"].map(month_map)

    # Cyclical encoding of month captures monsoon seasonality
    df["Month_sin"] = np.sin(2 * np.pi * df["MonthNum"] / 12) #add new columns to the dataframe for monthly sin and monthly cos
    df["Month_cos"] = np.cos(2 * np.pi * df["MonthNum"] / 12)

    feature_cols = [
        "Cloud Cover",
        "Diurnal Temp. Range",
        "Vapour Pressure",
        "Wet day Freq."
    ]

    target_col = "Precipitation"

    df = df.dropna(
        subset=feature_cols + [target_col]
    ).reset_index(drop=True)

    return df, feature_cols, target_col

Overwriting /content/data_prep.py


In [ ]:
import importlib
import data_prep

importlib.reload(data_prep)

import inspect
print(inspect.signature(data_prep.load_data))

(path='/content/Hyderabad_Consolidated_RA.xlsx')


In [ ]:
from data_prep import load_data

df, feature_cols, target_col = load_data()

print(df.shape)
print(feature_cols)
print(target_col)

(1224, 16)
['Cloud Cover', 'Diurnal Temp. Range', 'Vapour Pressure', 'Wet day Freq.']
Precipitation


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt

from data_prep import load_data
from models import build_drn_dsa_model, build_dsa_branch

tf.random.set_seed(42)
np.random.seed(42)

# ---------- 1. Load & split (chronological, like real forecasting) ----------
df, feature_cols, target_col = load_data()

X = df[feature_cols].values.astype("float32")
y_raw = df[target_col].values.astype("float32")
y = np.log1p(y_raw)  # stabilize heavy right skew; inverted for metrics later (convert precipitation into log values)

split = int(len(df) * 0.85)  # 90%-style split like the paper's "90% training"
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
y_raw_train, y_raw_test = y_raw[:split], y_raw[split:]

x_scaler = MinMaxScaler()
X_train_s = x_scaler.fit_transform(X_train)
X_test_s = x_scaler.transform(X_test)

n_features = X_train_s.shape[1]
print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}, features: {feature_cols}")

# ---------- 2. Build hybrid model ----------
model, dsa_autoencoder, dsa_encoder = build_drn_dsa_model(n_features, dsa_encoding_dims=(16, 8))

# ---------- 2a. Pre-train DSA layer-wise (unsupervised), paper Sec 3.3.3 ----------
# Warm up the encoder on the fused-feature-scale input by pretraining
# the standalone autoencoder on the raw scaled features first, then
# transferring learned encoder weights where dimensions allow is
# architecture-specific; here we pretrain the fused-space autoencoder
# directly using a quick projection so the "l2" space is learned before
# supervised fine-tuning.
tmp_fm = tf.keras.Input(shape=(n_features,))
tmp_proj = tf.keras.layers.Dense(16, activation="relu")(tmp_fm)
proj_model = tf.keras.Model(tmp_fm, tmp_proj)
l2_like_train = proj_model.predict(X_train_s, verbose=0)

dsa_autoencoder.compile(optimizer="adam", loss="mse")
dsa_autoencoder.fit(l2_like_train, l2_like_train, epochs=60, batch_size=32,
                     verbose=0, validation_split=0.1)
print("DSA pretraining (reconstruction) done.")

# ---------- 2b. Fine-tune full hybrid model end-to-end (supervised) ----------
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse")
history = model.fit(X_train_s, y_train, epochs=100, batch_size=32,
                     validation_split=0.1, verbose=0)
print("DRN-DSA fine-tuning done. Final train loss:", history.history["loss"][-1])

# ---------- 3. Predictions (invert log1p) ----------
pred_log = model.predict(X_test_s, verbose=0).flatten()
pred = np.expm1(pred_log)
pred = np.clip(pred, 0, None)

# ---------- 4. Baselines ----------
baselines = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Plain MLP": MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=2000, random_state=42),
}

results = {}


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    # MAPE, guard against zero precipitation months
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero]))
    rae = np.sqrt(np.sum((y_pred - y_true) ** 2)) / np.sqrt(np.sum(y_true ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot
    return dict(MSE=mse, RMSE=rmse, MAPE=mape, RAE=rae, R2=r2)


for name, mdl in baselines.items():
    mdl.fit(X_train_s, y_raw_train)
    p = mdl.predict(X_test_s)
    p = np.clip(p, 0, None)
    results[name] = metrics(y_raw_test, p)

results["DRN-DSA (proposed)"] = metrics(y_raw_test, pred)

res_df = pd.DataFrame(results).T
res_df = res_df[["MSE", "RMSE", "MAPE", "RAE", "R2"]]
print("\n=== Test-set performance ===")
print(res_df.round(4).to_string())

res_df.round(4).to_csv("results_table.csv", index=False)

# ---------- 5. Plots ----------
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("MSE (log-precip space)")
plt.title("DRN-DSA fine-tuning curve")
plt.legend()
plt.tight_layout()
plt.savefig("/content/training_curve.png", dpi=150)
plt.close()

plt.figure(figsize=(12, 4))
idx = np.arange(len(y_raw_test))
plt.plot(idx, y_raw_test, label="Ground truth", linewidth=1.5)
plt.plot(idx, pred, label="DRN-DSA prediction", linewidth=1.2)
plt.xlabel("Test month index")
plt.ylabel("Precipitation (mm)")
plt.title("Predicted vs actual monthly precipitation (test set)")
plt.legend()
plt.tight_layout()
plt.savefig("/content/prediction_vs_actual.png", dpi=150)
plt.close()

print("\nSaved: results_table.csv, training_curve.png, prediction_vs_actual.png")


Train: (1040, 4), Test: (184, 4), features: ['Cloud Cover', 'Diurnal Temp. Range', 'Vapour Pressure', 'Wet day Freq.']


DSA pretraining (reconstruction) done.
DRN-DSA fine-tuning done. Final train loss: 0.17421963810920715

=== Test-set performance ===
                         MSE     RMSE    MAPE     RAE      R2
Linear Regression   994.0588  31.5287  1.0141  0.2526  0.8979
Random Forest       946.2164  30.7606  0.5045  0.2465  0.9028
Plain MLP           967.5965  31.1062  0.8425  0.2492  0.9006
DRN-DSA (proposed)  786.6009  28.0464  0.4354  0.2247  0.9192

Saved: results_table.csv, training_curve.png, prediction_vs_actual.png


### **FDnet**

In [ ]:
"""
Turns the monthly Hyderabad data into (lookback_window -> next_month_precip)
samples, the format FDNet-style forecasting actually needs (unlike DRN-DSA,
which took a single month's snapshot).
"""

import numpy as np
from data_prep import load_data


def build_sequences(lookback=24, horizon=1):
    df, feature_cols, target_col = load_data()

    # Channels: the 4 correlated variables + precipitation's own history
    # (autoregressive signal helps short lookback windows a lot)
    channel_cols = feature_cols + [target_col]
    data = df[channel_cols].values.astype("float32")

    n = len(data)
    X, y = [], []
    for i in range(n - lookback - horizon + 1):
        X.append(data[i:i + lookback, :])                      # (lookback, channels)
        y.append(data[i + lookback:i + lookback + horizon, -1])  # future precip only
    X = np.stack(X)
    y = np.stack(y).squeeze(-1) if horizon == 1 else np.stack(y)
    return X, y, channel_cols


if __name__ == "__main__":
    X, y, cols = build_sequences(lookback=24, horizon=1)
    print("X:", X.shape, "y:", y.shape, "channels:", cols)


X: (1200, 24, 5) y: (1200,) channels: ['Cloud Cover', 'Diurnal Temp. Range', 'Vapour Pressure', 'Wet day Freq.', 'Precipitation']


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt

from sequence_prep import build_sequences
from fdnet_model import build_fdnet

tf.random.set_seed(42)
np.random.seed(42)

LOOKBACK = 24
PYRAMID = 4

# ---------- 1. Build sequences, chronological split ----------
X, y_raw, channel_cols = build_sequences(lookback=LOOKBACK, horizon=1)
y = np.log1p(y_raw)

split = int(len(X) * 0.85)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
y_raw_train, y_raw_test = y_raw[:split], y_raw[split:]

# scale each channel using train statistics only
n_channels = X.shape[-1]
scalers = []
X_train_s = np.zeros_like(X_train)
X_test_s = np.zeros_like(X_test)
for c in range(n_channels):
    sc = MinMaxScaler()
    X_train_s[:, :, c] = sc.fit_transform(X_train[:, :, c])
    X_test_s[:, :, c] = sc.transform(X_test[:, :, c])
    scalers.append(sc)

print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}, channels: {channel_cols}")

# ---------- 2. Build & train FDNet ----------
fdnet, sub_lengths = build_fdnet(lookback=LOOKBACK, n_channels=n_channels,
                                  pred_len=1, pyramid=PYRAMID, d_model=16)
print("Focal sub-sequence lengths (farthest -> nearest):", sub_lengths)

fdnet.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse")
history = fdnet.fit(X_train_s, y_train, epochs=100, batch_size=32,
                     validation_split=0.1, verbose=0)
print("FDNet training done. Final train loss:", history.history["loss"][-1])

pred_log = fdnet.predict(X_test_s, verbose=0).flatten()
pred = np.clip(np.expm1(pred_log), 0, None)

# ---------- 3. Baselines on the SAME windowed task (flatten window as features) ----------
X_train_flat = X_train_s.reshape(len(X_train_s), -1)
X_test_flat = X_test_s.reshape(len(X_test_s), -1)

baselines = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Plain MLP": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=42),
}


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero]))
    rae = np.sqrt(np.sum((y_pred - y_true) ** 2)) / np.sqrt(np.sum(y_true ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot
    return dict(MSE=mse, RMSE=rmse, MAPE=mape, RAE=rae, R2=r2)


results = {}
for name, mdl in baselines.items():
    mdl.fit(X_train_flat, y_raw_train)
    p = np.clip(mdl.predict(X_test_flat), 0, None)
    results[name] = metrics(y_raw_test, p)

results["FDNet (focal decomposed)"] = metrics(y_raw_test, pred)

res_df = pd.DataFrame(results).T[["MSE", "RMSE", "MAPE", "RAE", "R2"]]
print("\n=== Test-set performance (lookback=24 months) ===")
print(res_df.round(4).to_string())
res_df.round(4).to_csv("/content/results_table_fdnet.csv")

# ---------- 4. Plots ----------
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch"); plt.ylabel("MSE (log-precip space)")
plt.title("FDNet training curve")
plt.legend(); plt.tight_layout()
plt.savefig("/content/fdnet_training_curve.png", dpi=150)
plt.close()

plt.figure(figsize=(12, 4))
idx = np.arange(len(y_raw_test))
plt.plot(idx, y_raw_test, label="Ground truth", linewidth=1.5)
plt.plot(idx, pred, label="FDNet prediction", linewidth=1.2)
plt.xlabel("Test month index"); plt.ylabel("Precipitation (mm)")
plt.title("FDNet: predicted vs actual monthly precipitation (test set)")
plt.legend(); plt.tight_layout()
plt.savefig("/content/fdnet_prediction_vs_actual.png", dpi=150)
plt.close()

print("\nSaved: results_table_fdnet.csv, fdnet_training_curve.png, fdnet_prediction_vs_actual.png")


Train: (1020, 24, 5), Test: (180, 24, 5), channels: ['Cloud Cover', 'Diurnal Temp. Range', 'Vapour Pressure', 'Wet day Freq.', 'Precipitation']
Focal sub-sequence lengths (farthest -> nearest): [12, 6, 3, 3]
FDNet training done. Final train loss: 0.4142306447029114

=== Test-set performance (lookback=24 months) ===
                                MSE     RMSE     MAPE     RAE      R2
Linear Regression         4607.8663  67.8813  11.8970  0.5404  0.5352
Random Forest             3886.3033  62.3402   6.9115  0.4963  0.6080
Plain MLP                 3724.9880  61.0327   5.7988  0.4858  0.6243
FDNet (focal decomposed)  6607.4714  81.2864   5.8719  0.6471  0.3335

Saved: results_table_fdnet.csv, fdnet_training_curve.png, fdnet_prediction_vs_actual.png
